### source delta table

In [0]:
%sql
create table delta_catalog.raw.streamsrc(
  id int,
  order_name string,
  amount double,
  prod_id int
)
using delta
location 'abfss://raw@storagedeltalakepoorna.dfs.core.windows.net/streansrc'

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-5788397051607957>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', "create table delta_catalog.raw.streamsrc(\n  id int,\n  order_name string,\n  amount double,\n  prod_id int\n)\nusing delta\nlocation 'abfss://raw@storagedeltalakepoorna.dfs.core.windows.net/streansrc'\n")

File /databricks/python/lib/python3.12/site-packages/IPython/core/interactiveshell.py:2541, in InteractiveShell.run_cell_magic(self, magic_name, line, cell)
   2539 with self.builtin_trap:
   2540     args = (magic_arg_s, cell)
-> 2541     result = fn(*args, **kwargs)
   2543 # The code below prevents the output from being displayed
   2544 # when using magics with decorator @output_can_be_silenced
   2545 # when the last Python token in the expression is a ';'.
   2546 if getattr(fn, magic.MAGIC_OUTPUT_CAN_BE_SILENCED, False):

File /databricks/

In [0]:
%sql
insert into delta_catalog.raw.streamsrc
values (4, 'order1', 1001, 100),
       (5, 'order2', 2002, 200),
       (6, 'order3', 3003, 300)

num_affected_rows,num_inserted_rows
3,3


### streaming query

In [0]:
df=spark.readStream.table('delta_catalog.raw.streamsrc')

In [0]:
df.writeStream.format("delta")\
    .option("checkpointLocation", "abfss://raw@storagedeltalakepoorna.dfs.core.windows.net/streamsink/checkpoint")\
    .trigger(processingTime="10 seconds")\
    .start("abfss://raw@storagedeltalakepoorna.dfs.core.windows.net/streamsink/data")

In [0]:
%sql
select * from delta.`abfss://raw@storagedeltalakepoorna.dfs.core.windows.net/streamsink/data`

id,order_name,amount,prod_id
1,order1,1001.0,100
2,order2,2002.0,200
3,order3,3003.0,300
4,order1,1001.0,100
5,order2,2002.0,200
6,order3,3003.0,300
